In [1]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"



import jax
print(jax.devices())

jax.config.update("jax_enable_x64", True)
import numpy as np
import jax.numpy as jnp

import sys 

sys.path.append("/home/joshua/PhD_year_1/jaxsp/Adding_stellar_masses/")

import jaxsp as jsp

from jaxsp.constants import GN, hbar

from collections import defaultdict

import s2fft

from scipy.special import sph_harm_y

[CudaDevice(id=0)]


## Want to skip the spherical harmonic transform by using the fact that rho is already made up of squared sum of $Y_l^m$'s

In [2]:
m22 = 1
u = jsp.set_schroedinger_units(m22)

In [3]:
cNFWtides_params = jnp.array([
357964808.148399 * u.from_Msun,
25.690207,
0.407461,
0.012670 * u.from_Kpc,
1.857991 * u.from_Kpc,
3.729259
])

density_params = jsp.init_core_NFW_tides_params_from_sample(cNFWtides_params)

N = 512
rmin = .1 * u.from_pc
rmax = jsp.enclosing_radius(0.999, density_params)
potential_params = jsp.init_potential_params(density_params, rmin, rmax, N)

eval_library = jax.vmap(jax.vmap(jsp.eval_radial_eigenmode, in_axes=(None, 0)), in_axes=(0,None))

N = 1024
a = 1
b = 10

rmax = jsp.enclosing_radius(0.99, density_params)
eigenstate_lib = jsp.init_eigenstate_library(potential_params, rmin, rmax, a, b, N)

n = eigenstate_lib.radial_eigenmode_params.n
l = eigenstate_lib.radial_eigenmode_params.l
eigen_energies = eigenstate_lib.radial_eigenmode_params.E

print('l max from jaxsp:', max(l))
L = int(max(l) + 1)

rmin = 20 * u.from_pc

r = jnp.logspace(jnp.log10(rmin), jnp.log10(rmax), 1000)


R_j_r = eval_library(r, eigenstate_lib.radial_eigenmode_params)


tol = 1e-7
wavefunction_params = jsp.init_wavefunction_params(eigenstate_lib, density_params, rmin, rmax, tol)

total_mass = wavefunction_params.total_mass
aj_2 = wavefunction_params.aj_2        # shape (Nj,)
rand_phase = jax.random.uniform(jax.random.PRNGKey(0), shape=aj_2.shape, minval=0.0, maxval=2 * jnp.pi,)
aj = jnp.sqrt(aj_2) * jnp.exp(1j * rand_phase)  # shape (Nj,)


l max from jaxsp: 23


In [21]:
lm_l = []      # list of l for each mode k
lm_m = []      # list of m for each mode k
parent_j = []  # which radial eigenstate j this (l,m) mode comes from
lm_pairs = defaultdict(int)

l = eigenstate_lib.radial_eigenmode_params.l
n = eigenstate_lib.radial_eigenmode_params.n
print(l)
print(n)
print(len(l))

for j_idx, ell in enumerate(l.tolist()):
    for m in range(-ell, ell + 1):
        lm_l.append(ell)
        lm_m.append(m)
        parent_j.append(j_idx)
        lm_pairs[(ell, m)] += 1

lm_pairs = list(lm_pairs.keys()) # list of ((l,m), count) pairs

lm_pairs = jnp.array(lm_pairs)  # shape (Nmodes, 2)


'''Precompute Y_lm's for wavefunction reconstruction'''

# McEwen-Wiaux-style equiangular grid

L = int(max(l))+1
L_max_out = 2 * L - 1   # captures all density harmonics up to l1+l2 <= 2*(L-1)
n_theta = L_max_out
n_phi = 2 * L_max_out - 1

# Generate theta values
i = jnp.arange(n_theta)
theta = (jnp.pi * (2 * i + 1)) / (2 * L_max_out - 1)

# Generate phi values
j = jnp.arange(n_phi)
phi = (2 * jnp.pi * j) / (2 * L_max_out - 1)

Theta, Phi = jnp.meshgrid(theta, phi, indexing="ij")  # both (n_theta, n_phi)

Y_list = []
for ell, m in zip(lm_l, lm_m):
    Y_lm_mode = sph_harm_y(ell, m, Theta, Phi)  # (n_theta, n_phi), complex
    Y_list.append(Y_lm_mode)

Y_lm = jnp.stack(Y_list, axis=0)  # (Nmodes, n_theta, n_phi), complex

parent_j = jnp.array(parent_j)  # shape (Nmodes,)


[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1
  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  2  2  2
  2  2  2  2  2  2  2  2  2  2  2  2  2  2  2  2  2  2  3  3  3  3  3  3
  3  3  3  3  3  3  3  3  3  3  3  3  3  3  4  4  4  4  4  4  4  4  4  4
  4  4  4  4  4  4  4  4  4  5  5  5  5  5  5  5  5  5  5  5  5  5  5  5
  5  5  5  6  6  6  6  6  6  6  6  6  6  6  6  6  6  6  6  6  7  7  7  7
  7  7  7  7  7  7  7  7  7  7  7  7  8  8  8  8  8  8  8  8  8  8  8  8
  8  8  8  9  9  9  9  9  9  9  9  9  9  9  9  9  9 10 10 10 10 10 10 10
 10 10 10 10 10 10 10 11 11 11 11 11 11 11 11 11 11 11 11 11 12 12 12 12
 12 12 12 12 12 12 12 12 13 13 13 13 13 13 13 13 13 13 13 14 14 14 14 14
 14 14 14 14 14 15 15 15 15 15 15 15 15 15 16 16 16 16 16 16 16 16 17 17
 17 17 17 17 17 18 18 18 18 18 18 19 19 19 19 19 20 20 20 20 21 21 21 22
 22 23]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22  0
  1  2  3  4  5  6  7  8  9 10 11 12 13 14 

In [5]:
time_step = 0
total_time = 5 * u.to_Gyr
no_of_time_steps = 1000
dt = total_time / no_of_time_steps

radial_eigen_function_time_stepped = R_j_r * jnp.exp(-1j * time_step * dt * eigen_energies / hbar.value)


R_modes = radial_eigen_function_time_stepped[:, parent_j]
aj_modes = aj[jnp.array(parent_j)]

full_psi_rtp = jnp.einsum('k,rk,ktp->rtp', aj_modes, R_modes, Y_lm)

psi_abs2 = jnp.abs(full_psi_rtp) ** 2
rho_rtp = total_mass * psi_abs2

## Performing S2FFT

In [6]:
def forward_sht_single_r(rho_at_r):
    return s2fft.forward(rho_at_r, L_max_out, sampling='mw', method='jax')

rho_lm_r = jax.vmap(forward_sht_single_r)(rho_rtp)  # (Nr, L, 2*L-1)

print('rho_lm_r shape:', rho_lm_r.shape)

rho_lm_r shape: (1000, 47, 93)


## Detailed Derivation

### Step 1: The Wavefunction Expansion

The wavefunction is written as a mode expansion. Each mode $k$ is labeled by a radial index $j(k)$ and angular momentum numbers $(l_k, m_k)$. The expansion separates radial and angular parts:

$$\psi(r,\theta,\phi) = \sum_k a_k\, R_{j(k)}(r)\, Y_{l_k}^{m_k}(\theta,\phi)$$

- $R_{j(k)}(r)$ — radial eigenfunction (e.g. spherical Bessel function or Schrödinger eigenmode)
- $Y_l^m(\theta,\phi)$ — spherical harmonics, the orthonormal angular basis on $S^2$
- $a_k \in \mathbb{C}$ — complex mode coefficients

---

### Step 2: Computing the Density

The physical density is $\rho = M|\psi|^2 = M\,\psi\,\psi^*$. Expanding the product:

$$\rho = M \left(\sum_k a_k R_{j(k)} Y_{l_k}^{m_k}\right)\!\left(\sum_{k'} a_{k'}^* R_{j(k')} \overline{Y_{l_{k'}}^{m_{k'}}}\right) = M \sum_{k,k'} a_k a_{k'}^*\, R_{j(k)} R_{j(k')}\, Y_{l_k}^{m_k}\, \overline{Y_{l_{k'}}^{m_{k'}}}$$

This is exact — a double sum over all mode pairs $(k, k')$.

---

### Step 3: Projecting onto Spherical Harmonics

The spherical harmonic coefficients of $\rho$ at radius $r$ are defined by:

$$\rho_{LM}(r) = \int_{S^2} \rho(r,\theta,\phi)\, \overline{Y_L^M(\theta,\phi)}\; d\Omega, \qquad d\Omega = \sin\theta\,d\theta\,d\phi$$

Substituting the double sum and pulling $R_{j(k)}(r)$, $R_{j(k')}(r)$ outside the angular integral:

$$\rho_{LM}(r) = M \sum_{k,k'} a_k a_{k'}^*\, R_{j(k)}(r)\, R_{j(k')}(r) \underbrace{\int Y_{l_k}^{m_k}\, \overline{Y_{l_{k'}}^{m_{k'}}}\, \overline{Y_L^M}\; d\Omega}_{G(l_k,m_k,\,l_{k'},m_{k'},\,L,M)}$$

The remaining integral over three spherical harmonics is the **Gaunt coefficient** $G$.

---

### Step 4: Evaluating the Gaunt Coefficient

We need:

$$G(l_1,m_1,l_2,m_2,L,M) = \int Y_{l_1}^{m_1}\, \overline{Y_{l_2}^{m_2}}\, \overline{Y_L^M}\; d\Omega$$

**Convert conjugates** using $\overline{Y_l^m} = (-1)^m Y_l^{-m}$:

$$G = (-1)^{m_2+M} \int Y_{l_1}^{m_1}\, Y_{l_2}^{-m_2}\, Y_L^{-M}\; d\Omega$$

**Apply the triple-harmonic integral formula** (standard result from angular momentum theory):

$$\int Y_{l_1}^{m_1} Y_{l_2}^{m_2} Y_{l_3}^{m_3}\, d\Omega = \sqrt{\frac{(2l_1+1)(2l_2+1)(2l_3+1)}{4\pi}} \begin{pmatrix}l_1 & l_2 & l_3 \\ 0 & 0 & 0\end{pmatrix}\begin{pmatrix}l_1 & l_2 & l_3 \\ m_1 & m_2 & m_3\end{pmatrix}$$

Substituting $(m_2 \to -m_2,\; m_3 \to -M)$ and absorbing the sign:

$$\boxed{G(l_1,m_1,l_2,m_2,L,M) = (-1)^{m_2+M} \sqrt{\frac{(2l_1+1)(2l_2+1)(2L+1)}{4\pi}} \begin{pmatrix}l_1 & l_2 & L \\ 0 & 0 & 0\end{pmatrix}\begin{pmatrix}l_1 & l_2 & L \\ m_1 & {-m_2} & {-M}\end{pmatrix}}$$

where $\begin{pmatrix}\cdot & \cdot & \cdot \\ \cdot & \cdot & \cdot\end{pmatrix}$ denotes a Wigner 3j symbol.

---

### Step 5: Selection Rules

The 3j symbols vanish unless the following conditions all hold simultaneously:

| Rule | Origin |
|------|--------|
| $M = m_1 - m_2$ | Lower row of second 3j symbol must sum to zero: $m_1 + (-m_2) + (-M) = 0$ |
| $\|l_1 - l_2\| \leq L \leq l_1 + l_2$ | Triangle inequality on angular momentum addition |
| $l_1 + l_2 + L$ even | Parity symmetry — required by the first 3j symbol with $0,0,0$ bottom row |

These make $G$ extremely sparse: most $(k,k',L,M)$ combinations contribute exactly zero.

---

### Step 6: The Optimisation

**Problem:** the naive double sum over all $N_\text{modes}$ modes is $O(N_\text{modes}^2)$. Many modes share the same $(l,m)$ but differ only in their radial index $j$ — work is being repeated.

**Key insight:** group all modes with identical angular quantum numbers $(l,m)$, and define a single combined radial function for each unique pair:

$$F_{l,m}(r) = \sum_{\substack{k:\,(l_k,m_k)=(l,m)}} a_k\, R_{j(k)}(r)$$

Because $G$ depends only on $(l_1, m_1, l_2, m_2, L, M)$ and not on the individual radial index $j$, we can factor it outside the inner sums over $j$:

$$\rho_{LM}(r) = M \sum_{(l_1,m_1)}\sum_{(l_2,m_2)} G(l_1,m_1,l_2,m_2,L,M)\; F_{l_1,m_1}(r)\; F_{l_2,m_2}^*(r)$$

The double sum now runs over **unique $(l,m)$ pairs** only. Since $|m| \leq l \leq l_\text{max}$, there are at most $(l_\text{max}+1)^2$ unique pairs, giving:

$$O(N_\text{modes}^2) \;\longrightarrow\; O(N_\text{unique}^2) \leq O\!\left((l_\text{max}+1)^4\right)$$

This is a strict improvement whenever multiple radial modes share the same $(l,m)$ — which is always the case here since the radial and angular quantum numbers are chosen independently.

In [30]:

from collections import defaultdict


import numpy as np
import wigners
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

def _process_l1l2(args):
    l1, l2, L_max_out = args
    parts = []
    L_min = abs(l1 - l2)
    L_max = min(l1 + l2, L_max_out - 1)
    for L in range(L_min, L_max + 1):
        if (l1 + l2 + L) % 2 != 0:
            continue
        w0 = wigners.wigner_3j(l1, l2, L, 0, 0, 0)
        if w0 == 0.0:
            continue
        prefac0 = np.sqrt((2*l1+1)*(2*l2+1)*(2*L+1) / (4*np.pi)) * w0

        arr = wigners.clebsch_gordan_array(l1, l2, L)  # (2l1+1, 2l2+1, 2L+1)
        m1g, m2g = np.meshgrid(np.arange(-l1, l1+1), np.arange(-l2, l2+1), indexing='ij')
        Mg = m1g - m2g
        valid = np.abs(Mg) <= L
        m1v, m2v, Mv = m1g[valid], m2g[valid], Mg[valid]

        neg_m2_idx = (-m2v) + l2
        M_idx_L    = Mv + L
        cg_vals    = arr[m1v + l1, neg_m2_idx, M_idx_L]

        phase = ((-1.0) ** (l1 - l2 + Mv + m2v + Mv)) / np.sqrt(2*L+1)
        G_vals = prefac0 * phase * cg_vals

        nz = np.abs(G_vals) > 1e-14
        if not np.any(nz):
            continue

        parts.append((l1, m1v[nz], l2, m2v[nz], G_vals[nz],
                      L * (2*L_max_out - 1) + (Mv[nz] + L_max_out - 1)))
    return parts


def precompute_gaunt_table(lm_l, lm_m, L_max_out, cache_dir=".", n_workers=64):
    lm_l = list(map(int, lm_l))
    lm_m = list(map(int, lm_m))
    unique_lm = sorted(set(zip(lm_l, lm_m)))
    lm_to_idx = {lm: i for i, lm in enumerate(unique_lm)}
    L_max = max(l for l, m in unique_lm)

    # --- disk cache ---
    cache_path = Path(cache_dir) / f"gaunt_lmax{L_max}_Lout{L_max_out}.npz"
    if cache_path.exists():
        print(f"Loading Gaunt table from {cache_path} ...")
        data = np.load(cache_path)
        return (jnp.array(data["i"]), jnp.array(data["j"]), jnp.array(data["G"]),
                jnp.array(data["Lf"]), unique_lm)
    

    # --- build ---
    print(f"Building Gaunt table ({len(unique_lm)} unique (l,m), L_max_out={L_max_out}) ...")
    tasks = [(l1, l2, L_max_out) for l1 in range(L_max+1) for l2 in range(L_max+1)]
    
    all_i, all_j, all_G, all_Lf = [], [], [], []
    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        for parts in pool.map(_process_l1l2, tasks, chunksize=4):
            for l1, m1v, l2, m2v, G_vals, Lf_vals in parts:
                all_i.append([lm_to_idx[(l1, int(m))] for m in m1v])
                all_j.append([lm_to_idx[(l2, int(m))] for m in m2v])
                all_G.append(G_vals)
                all_Lf.append(Lf_vals)

    ai = np.concatenate(all_i).astype(np.int32)
    aj = np.concatenate(all_j).astype(np.int32)
    aG = np.concatenate(all_G)
    aLf = np.concatenate(all_Lf).astype(np.int32)

    np.savez(cache_path, i=ai, j=aj, G=aG, Lf=aLf)
    print(f"  {len(ai):,} non-zero entries — saved to {cache_path}")
    return jnp.array(ai), jnp.array(aj), jnp.array(aG), jnp.array(aLf), unique_lm


def compute_rho_lm_gaunt(aj_modes, R_modes, lm_l, lm_m, total_mass, time_step,
                          L_max_out=None, gaunt_table=None, batch_size=100_000):
    """
    Compute rho_LM(r) using Gaunt coefficients — no grid, no SHT.

    Speedups vs naive version
    -------------------------
    1. Modes sharing (l,m) combined into F_{l,m}(r) first, reducing the
       double sum from O(N_modes^2) to O(N_unique^2).
    2. Gaunt table precomputed once and passed in via gaunt_table so it can
       be reused across many time steps at no extra cost.
    3. Accumulation is a single JIT-compiled segment_sum on GPU per batch —
       ~10 Python iterations instead of ~2209.

    Parameters
    ----------
    aj_modes   : complex (Nmodes,)    per-mode a_k; rand_phase included
    R_modes    : complex (Nr, Nmodes) R_{j(k)}(r) * exp(-i E_k t/hbar)
    lm_l, lm_m : int array-like (Nmodes,)
    total_mass : float
    L_max_out  : int, optional  default 2*l_max+1 captures all density harmonics
    gaunt_table : tuple, optional  from precompute_gaunt_table(); computed here if None
    batch_size  : int  Gaunt entries per GPU batch — tune to available VRAM
                       (batch_size=100_000 ≈ 1.6 GB complex128 for Nr=1000)

    Returns
    -------
    rho_lm : JAX complex array (Nr, L_max_out, 2*L_max_out-1)
        rho_lm[r, L, L_max_out-1+M] = rho_LM(r)
        Same indexing convention as s2fft.forward (mw sampling).
    """
    Nr, Nmodes = R_modes.shape
    lm_l = list(map(int, lm_l))
    lm_m = list(map(int, lm_m))
    l_max = max(lm_l)

    if L_max_out is None:
        L_max_out = 2 * l_max + 1
    N_flat = L_max_out * (2 * L_max_out - 1)

    # ── 1. aR[r,k] = a_k * R_k(r)  (time phase already baked into R_modes) ───
    aR = np.array(R_modes, dtype=complex) * np.array(aj_modes, dtype=complex)[None, :] * np.exp(-1j * time_step * dt * eigen_energies[parent_j] / hbar.value)[None, :]
    print(R_modes.shape, aj_modes.shape)
    print(aR.shape)

    # ── 2. F_{l,m}(r) = sum_{k:(l_k,m_k)=(l,m)} a_k R_k(r) ──────────────────
    lm_to_F = defaultdict(lambda: np.zeros(Nr, dtype=complex))
    for k, (ell, m) in enumerate(zip(lm_l, lm_m)):
        lm_to_F[(ell, m)] += aR[:, k]

    # ── 3. Gaunt table (build once, reuse across time steps) ──────────────────
    if gaunt_table is None:
        gaunt_table = precompute_gaunt_table(lm_l, lm_m, L_max_out)

    all_i, all_j, all_G, all_Lf, unique_lm = gaunt_table
    N_nz = len(all_i)
    print(f"{N_nz:,} non-zero Gaunt entries to accumulate")

    # F_jax[r, i] = F_{unique_lm[i]}(r)  on GPU
    F_jax = jnp.array(np.stack([lm_to_F[lm] for lm in unique_lm], axis=1))  # (Nr, N_unique)

    # ── 4. JIT-compiled batch accumulation on GPU ──────────────────────────────
    # For each entry e:  weighted[e, r] = G_e * F_{l1,m1}(r) * conj(F_{l2,m2}(r))
    # segment_sum groups by flat (L,M) index → (N_flat, Nr)
    @jax.jit
    def _accum_batch(F, i_b, j_b, G_b, Lf_b):
        weighted = (F[:, i_b] * jnp.conj(F[:, j_b]) * G_b[None, :]).T   # (B, Nr)
        return jax.ops.segment_sum(weighted, Lf_b, num_segments=N_flat)   # (N_flat, Nr)

    rho_flat = jnp.zeros((N_flat, Nr), dtype=complex)
    n_batches = (N_nz + batch_size - 1) // batch_size
    for b, start in enumerate(range(0, N_nz, batch_size)):
        end = min(start + batch_size, N_nz)
        rho_flat = rho_flat + _accum_batch(
            F_jax, all_i[start:end], all_j[start:end],
            all_G[start:end], all_Lf[start:end],
        )
        print(f"\r  batch {b+1}/{n_batches}", end="", flush=True)
    print()

    # ── 5. (N_flat, Nr).T → (Nr, L_max_out, 2*L_max_out-1) ───────────────────
    return total_mass * rho_flat.T.reshape(Nr, L_max_out, 2 * L_max_out - 1)


In [31]:
L_max_out = 2 * L - 1  # captures all density harmonics up to l1+l2 <= 2*(L-1)
                         # use L_max_out=L to match s2fft output shape exactly


# Precompute Gaunt table ONCE — reuse this across all time steps
gaunt_table = precompute_gaunt_table(lm_l, lm_m, L_max_out)

time_step = 0
# Then per time step, only compute_rho_lm_gaunt (fast)
rho_lm_gaunt = compute_rho_lm_gaunt(
    aj_modes, R_modes, lm_l, lm_m, total_mass, time_step,
    L_max_out=L_max_out,
    
    gaunt_table=gaunt_table,
)
print("rho_lm_gaunt shape:", rho_lm_gaunt.shape)


Loading Gaunt table from gaunt_lmax23_Lout47.npz ...
(1000, 4800) (4800,)
(1000, 4800)
3,292,688 non-zero Gaunt entries to accumulate


  batch 33/33
rho_lm_gaunt shape: (1000, 47, 93)


In [9]:
print(rho_lm_gaunt.shape)
print(rho_lm_r.shape)

# recreate rho_rtp to see which is more accurate


lm_l_gaunt = []
lm_m_gaunt = []
for l in range(rho_lm_gaunt.shape[1]):
    for m in range(-l, l+1):
        lm_l_gaunt.append(l)
        lm_m_gaunt.append(m)


def inverse_sht_single_r_gaunt(rho_lm_gaunt_r):
    rho_rtp_recon = jnp.zeros_like(rho_rtp[0])
    for (l, m) in zip(lm_l_gaunt, lm_m_gaunt):
        Y_lm_mode = sph_harm_y(l, m, Theta, Phi)
        rho_rtp_recon += rho_lm_gaunt_r[l, m + L_max_out - 1] * Y_lm_mode
    return rho_rtp_recon

rho_rtp_gaunt = jax.vmap(inverse_sht_single_r_gaunt)(rho_lm_gaunt)  # (Nr, L, 2*L-1)


error_gaunt_recon = jnp.linalg.norm(rho_rtp - rho_rtp_gaunt) / jnp.linalg.norm(rho_rtp)
print(f"error of Gaunt reconstruction: {error_gaunt_recon:.2e}")

def inverse_sht_single_r(rho_lm_r):
    return s2fft.inverse(rho_lm_r, L_max_out, sampling='mw', method='jax')

rho_rtp_s2fft = jax.vmap(inverse_sht_single_r)(rho_lm_r)  # (Nr, L, 2*L-1)

error_s2fft_recon = jnp.linalg.norm(rho_rtp - rho_rtp_s2fft) / jnp.linalg.norm(rho_rtp)
print(f"error of s2fft reconstruction: {error_s2fft_recon:.2e}")



(1000, 47, 93)
(1000, 47, 93)
error of Gaunt reconstruction: 2.65e-15
error of s2fft reconstruction: 2.80e-14
